In [21]:
from pyspark.sql import SparkSession
import subprocess
import pandas as pd
from pyspark.sql.functions import col, lit
import psycopg2
from pyspark.sql.functions import monotonically_increasing_id, lit



spark = SparkSession.builder \
    .appName("RetailLakehouse_GoldToPostgres") \
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.1.0,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "org.postgresql:postgresql:42.7.3") \
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint",          "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",        "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key",        "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Session active: {spark.sparkContext.appName}")

Spark version : 3.5.0
Session active: RetailLakehouse_GoldToPostgres


In [22]:
# ─────────────────────────────────────────
# JDBC CONNECTION CONFIG
# ─────────────────────────────────────────
JDBC_URL = "jdbc:postgresql://month1-postgres-retail-1:5432/retaildw"

JDBC_PROPS = {
    "user":     "dataeng",
    "password": "dataeng123",
    "driver":   "org.postgresql.Driver"
}

print(f"JDBC URL: {JDBC_URL}")

JDBC URL: jdbc:postgresql://month1-postgres-retail-1:5432/retaildw


In [23]:
result = subprocess.run(
    ["pip", "install", "psycopg2-binary", "--quiet"],
    capture_output=True, text=True
)
print(result.stdout if result.stdout else "Installed")
print(result.stderr if result.stderr else "")

Installed



In [ ]:
# ─────────────────────────────────────────
# TRUNCATE ALL TABLES IN CORRECT ORDER
# Fact first, then dimensions
# ─────────────────────────────────────────

conn = psycopg2.connect(
    host="month1-postgres-retail-1",
    port=5432,
    dbname="retaildw",
    user="dataeng",
    password="dataeng123"
)
cur = conn.cursor()

# Must truncate fact first — it references dimensions
truncate_order = [
    "warehouse.fact_sales",
    "warehouse.dim_customer",
    "warehouse.dim_product",
    "warehouse.dim_store",
    "warehouse.dim_date"
]

for table in truncate_order:
    cur.execute(f"TRUNCATE TABLE {table} CASCADE;")
    print(f"Truncated {table}")

conn.commit()
cur.close()
conn.close()
print("\n All tables truncated — ready for load")

✔ Truncated warehouse.fact_sales
✔ Truncated warehouse.dim_customer
✔ Truncated warehouse.dim_product
✔ Truncated warehouse.dim_store
✔ Truncated warehouse.dim_date

✔ All tables truncated — ready for load


In [ ]:
date_range = pd.date_range(start="2024-01-01", end="2024-12-31", freq="D")

df_date_pd = pd.DataFrame({
    "full_date"      : date_range,
    "date_key"       : date_range.strftime("%Y%m%d").astype(int),
    "year_number"    : date_range.year,
    "quarter_number" : date_range.quarter,
    "month_number"   : date_range.month,
    "month_name"     : date_range.strftime("%B"),
    "day_of_week"    : date_range.dayofweek + 1,
    "day_name"       : date_range.strftime("%A"),
    "is_weekend"     : date_range.dayofweek >= 5,
    "is_holiday"     : False,
})

df_date = spark.createDataFrame(df_date_pd)

df_date.write.jdbc(
    url=JDBC_URL,
    table="warehouse.dim_date",
    mode="append",
    properties=JDBC_PROPS
)

print(f"dim_date loaded: {df_date.count():,} rows")

✔ dim_date loaded: 366 rows


In [ ]:
# ─────────────────────────────────────────
# LOAD DIMENSIONS TO POSTGRESQL
# ─────────────────────────────────────────

# DIM_CUSTOMER
df_gold_customers = spark.read.format("delta") \
    .load("s3a://gold/delta/customers")

df_gold_customers.select(
    col("customer_key"),
    col("customer_id"),
    col("customer_name"),
    col("email"),
    col("city"),
    col("state"),
    col("zip_code"),
    col("customer_segment"),
    col("effective_date").cast("date"),
    col("expiry_date").cast("date"),
    col("is_current")
).write.jdbc(
    url=JDBC_URL,
    table="warehouse.dim_customer",
    mode="append",
    properties=JDBC_PROPS
)
print(f"dim_customer loaded: {df_gold_customers.count():,} rows")

# DIM_PRODUCT
df_gold_products = spark.read.format("delta") \
    .load("s3a://gold/delta/products")

df_gold_products.select(
    col("product_key"),
    col("product_id"),
    col("product_name"),
    col("category"),
    col("subcategory"),
    col("brand"),
    col("list_price").cast("decimal(10,2)"),
    col("cost_price").cast("decimal(10,2)"),
    col("is_active")
).write.jdbc(
    url=JDBC_URL,
    table="warehouse.dim_product",
    mode="append",
    properties=JDBC_PROPS
)
print(f"dim_product loaded : {df_gold_products.count():,} rows")

# DIM_STORE
df_gold_stores = spark.read.format("delta") \
    .load("s3a://gold/delta/stores")

df_gold_stores.select(
    col("store_key"),
    col("store_id"),
    col("store_name"),
    col("city"),
    col("state"),
    col("region"),
    col("store_type"),
    col("opening_date").cast("date")
).write.jdbc(
    url=JDBC_URL,
    table="warehouse.dim_store",
    mode="append",
    properties=JDBC_PROPS
)
print(f"dim_store loaded   : {df_gold_stores.count():,} rows")

✔ dim_customer loaded: 10,000 rows
✔ dim_product loaded : 1,000 rows
✔ dim_store loaded   : 200 rows


In [ ]:
# ─────────────────────────────────────────
# LOAD FACT_SALES TO POSTGRESQL
# ─────────────────────────────────────────
df_gold_fact = spark.read.format("delta") \
    .load("s3a://gold/delta/fact_sales")

df_gold_customers_keys = spark.read.format("delta") \
    .load("s3a://gold/delta/customers") \
    .select("customer_id", "customer_key")

df_gold_products_keys = spark.read.format("delta") \
    .load("s3a://gold/delta/products") \
    .select("product_id", "product_key")

df_gold_stores_keys = spark.read.format("delta") \
    .load("s3a://gold/delta/stores") \
    .select("store_id", "store_key")

df_fact_final = df_gold_fact \
    .join(df_gold_customers_keys, "customer_id", "left") \
    .join(df_gold_products_keys,  "product_id",  "left") \
    .join(df_gold_stores_keys,    "store_id",    "left") \
    .withColumn("sales_key", monotonically_increasing_id().cast("bigint") + lit(1)) \
    .select(
        col("sales_key"),
        col("date_key"),
        col("customer_key"),
        col("product_key"),
        col("store_key"),
        col("order_id"),
        col("order_line_num"),
        col("quantity"),
        col("unit_price").cast("decimal(10,2)"),
        col("unit_cost").cast("decimal(10,2)"),
        col("discount_amount").cast("decimal(10,2)"),
        col("net_revenue").cast("decimal(12,2)"),
        col("gross_profit").cast("decimal(12,2)"),
        col("tax_amount").cast("decimal(10,2)")
    )

df_fact_final.write.jdbc(
    url=JDBC_URL,
    table="warehouse.fact_sales",
    mode="append",
    properties=JDBC_PROPS
)

print(f"fact_sales loaded: {df_fact_final.count():,} rows")

✔ fact_sales loaded: 500,000 rows


In [ ]:
# ─────────────────────────────────────────
# VERIFY POSTGRESQL LOAD
# ─────────────────────────────────────────
tables = [
    "warehouse.dim_date",
    "warehouse.dim_customer",
    "warehouse.dim_product",
    "warehouse.dim_store",
    "warehouse.fact_sales"
]

for table in tables:
    df = spark.read.jdbc(
        url=JDBC_URL,
        table=table,
        properties=JDBC_PROPS
    )
    print(f"{table}: {df.count():,} rows")

✔ warehouse.dim_date: 366 rows
✔ warehouse.dim_customer: 10,000 rows
✔ warehouse.dim_product: 1,000 rows
✔ warehouse.dim_store: 200 rows
✔ warehouse.fact_sales: 500,000 rows


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 44424)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =